# DINOv2 Zero-Shot Analysis — eVTOL Patents

**Purpose:** Extract structural embeddings from patent drawings using frozen DINOv2-large (no fine-tuning), cluster them with DBSCAN, and visualize with UMAP.

**Architecture:** All logic lives in [`src/zero_shot.py`](../src/zero_shot.py). This notebook configures paths/hyperparameters and calls one function per stage.

| Stage | What it does |
|---|---|
| 0 — Paths | Resolve input/output directories via `src.config_loader` |
| 1 — Config | Set hyperparameters; import from `src.zero_shot` |
| 2 — Extract | Batched CLS-token extraction + patent-level mean pooling |
| 3 — Normalize | L2-project embeddings onto the unit hypersphere |
| 4 — Cluster | DBSCAN with eps grid search; evaluate with Silhouette + Davies-Bouldin |
| 5 — Visualize | UMAP 2D scatter (seaborn) + per-cluster image gallery |

In [ ]:
# ── Stage 0: Path Configuration ──────────────────────────────────────────────
import sys
from pathlib import Path

# Make pipeline/ importable from notebooks/
sys.path.insert(0, str(Path("..").resolve()))

from src.config_loader import load_config
cfg = load_config()

IMAGE_DIR = Path(
    "/home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Drive_files_to_syncronize"
    "/3 - Images Processing & DataSets/2-Images_DataSets"
    "/521_Patents_DataSet/Main_AI_Approve_Images_from 187_patents"
)

DRIVE_PATH = cfg["paths"].get(
    "drive_root",
    Path("/home/vasco/Tese_Vasco_Lnx/Drive_files_to_syncronize"
         "/4 - Intelligence Models & Experimentation/1-Zero_shot_DINOv2"),
)

ANALYSIS_DIR  = DRIVE_PATH / "zero_shot_analysis"
OUTPUT_DIR    = ANALYSIS_DIR / "outputs"
EMBEDDING_DIR = OUTPUT_DIR / "embeddings"
PLOT_DIR      = OUTPUT_DIR / "plots"

for d in [ANALYSIS_DIR, OUTPUT_DIR, EMBEDDING_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Images  : {IMAGE_DIR}  (exists: {IMAGE_DIR.exists()})")
print(f"Outputs : {OUTPUT_DIR}")


In [ ]:
# ── Stage 1: Hyperparameters & Imports ───────────────────────────────────────
import torch
import matplotlib.pyplot as plt
from src.zero_shot import (
    collect_image_paths, initialize_dinov2, extract_embeddings,
    l2_normalize, dbscan_cluster, umap_project,
    plot_umap_clusters, plot_cluster_gallery,
    safe_save_np, safe_save_df, safe_save_plot,
)

SEED               = 42
MODEL_NAME         = "facebook/dinov2-large"   # 1024-d CLS token
BATCH_SIZE         = 16
DBSCAN_MIN_SAMPLES = 3                          # lower → more/larger clusters
DEVICE             = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Model: {MODEL_NAME}  |  Device: {DEVICE}  |  Seed: {SEED}")


In [ ]:
# ── Stage 2: Embedding Extraction ───────────────────────────────────────────
image_paths = collect_image_paths(IMAGE_DIR)
processor, model = initialize_dinov2(MODEL_NAME, DEVICE)

image_emb, img_meta_df, patent_ids, patent_emb = extract_embeddings(
    image_paths, processor, model, DEVICE, BATCH_SIZE
)

safe_save_np(patent_emb,  EMBEDDING_DIR / "patent_embeddings.npy")
safe_save_np(image_emb,   EMBEDDING_DIR / "image_embeddings.npy")
safe_save_df(img_meta_df, EMBEDDING_DIR / "image_metadata.csv")


In [ ]:
# ── Stage 3: L2 Normalization ─────────────────────────────────────────────────
X_norm = l2_normalize(patent_emb)


In [ ]:
# ── Stage 4: DBSCAN Clustering ───────────────────────────────────────────────
cluster_df, patent_cluster_labels, effective_eps = dbscan_cluster(
    X_norm, patent_ids, min_samples=DBSCAN_MIN_SAMPLES
)
saved = safe_save_df(cluster_df, OUTPUT_DIR / "patent_cluster_assignments_dbscan.csv")
print(f"✓ Assignments saved: {saved.name}")


In [ ]:
# ── Stage 5: UMAP Visualization ──────────────────────────────────────────────
points_2d = umap_project(X_norm, seed=SEED)

fig = plot_umap_clusters(
    points_2d, patent_cluster_labels, patent_ids, effective_eps, DBSCAN_MIN_SAMPLES
)
safe_save_plot(fig, PLOT_DIR / "patent_clusters_umap.png", dpi=220, bbox_inches="tight")
plt.show()

plot_cluster_gallery(cluster_df, patent_ids, image_paths, PLOT_DIR)
